In [1]:
import pandas as pd
import numpy as np

In [2]:
url="https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data"

In [4]:
cols=["Sex", "Length", "Diameter", "Height", "Whole_wt", "Shucked_wt", "Viscera_wt", "Shell_wt", "Rings"]
df=pd.read_csv(url, names=cols)

In [6]:
print(f"Number of rows:{len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(df.head())

Number of rows:4177
Columns: ['Sex', 'Length', 'Diameter', 'Height', 'Whole_wt', 'Shucked_wt', 'Viscera_wt', 'Shell_wt', 'Rings']
  Sex  Length  Diameter  Height  Whole_wt  Shucked_wt  Viscera_wt  Shell_wt  \
0   M   0.455     0.365   0.095    0.5140      0.2245      0.1010     0.150   
1   M   0.350     0.265   0.090    0.2255      0.0995      0.0485     0.070   
2   F   0.530     0.420   0.135    0.6770      0.2565      0.1415     0.210   
3   M   0.440     0.365   0.125    0.5160      0.2155      0.1140     0.155   
4   I   0.330     0.255   0.080    0.2050      0.0895      0.0395     0.055   

   Rings  
0     15  
1      7  
2      9  
3     10  
4      7  


In [7]:
df["y"]=df["Rings"]+1.5

In [12]:
feature_cols=["Length","Diameter","Whole_wt"]
X=df[feature_cols].values
y=df["y"].values
# 1. Length: Older abalones generally grow longer.
# 2. Diameter: They also get wider as they age.
# 3. Whole_wt: Mass is usually the strongest indicator of growth.

In [9]:
split_idx=int(len(X)*0.8)
X_train,X_test=X[:split_idx],X[split_idx:]
y_train,y_test=y[:split_idx],y[split_idx:]

In [10]:
print("Training set")
print(f"X:{X_train.shape}, y:{y_train.shape}")
print("Testing set")
print(f"X:{X_test.shape},  y:{y_test.shape}")


Training set
X:(3341, 3), y:(3341,)
Testing set
X:(836, 3),  y:(836,)


In [14]:
mean=X_train.mean(axis=0)
std=X_train.std(axis=0)
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std
# Without normalization, 'Weight' (which has large numbers) would overpower 'Length' (small numbers),
# and the model would think Weight is 100x more important just because of the units.

In [15]:
def forward(X, w, b):
    return X @ w + b
    # This is the linear equation: y = w1*x1 + w2*x2 ... + b
    # It calculates a weighted score for every row at once using Matrix Math (@).

In [17]:

d = X_train.shape[1]
print(f"X shape:{X_train.shape}")
print(f"w shape:({d},) -> One weight per feature")
print(f"b shape:Scalar -> Just one number")

X shape:(3341, 3)
w shape:(3,) -> One weight per feature
b shape:Scalar -> Just one number


In [18]:
def mse(y, y_hat):
    return np.mean((y - y_hat) ** 2)
    # Mean Squared Error.
    # We square the difference so negative errors (-5) don't cancel out positive ones (+5).
    # This also heavily penalizes outliers (big mistakes get punished much harder).

In [20]:
def grad_w(X, y, y_hat):
    n = len(y)
    error = y_hat - y
    return (2/n) * (X.T @ error)
     # The 'slope' for weights. It tells us how to adjust w to reduce error.

def grad_b(y, y_hat):
    n = len(y)
    error = y_hat - y
    return (2/n) * np.sum(error)
    # The 'slope' for bias.

    # What gradient means: It points UPHILL, towards higher error.
# Why we subtract it: We want to go DOWNHILL to the valley of lowest error.

In [21]:
np.random.seed(42) # For consistent results
w = np.random.randn(d) * 0.01
b = 0.0
lr = 0.1
epochs = 1000

In [22]:
for i in range(epochs):
    # 1. Forward Pass (Make a guess)
    y_hat = forward(X_train, w, b)

    # 2. Compute Loss (Check scorecard)
    loss = mse(y_train, y_hat)

    # 3. Compute Gradients (Find direction)
    dw = grad_w(X_train, y_train, y_hat)
    db = grad_b(y_train, y_hat)

    # 4. Update Weights (Take a step)
    w -= lr * dw
    b -= lr * db

    # Print progress to prove it's learning
    if i % 100 == 0:
        print(f"Epoch {i}: Loss = {loss:.4f}")


# Expectation: The loss started high and dropped fast, then settled down.
# This proves the machine is actually learning the pattern.

Epoch 0: Loss = 144.2636
Epoch 100: Loss = 7.4074
Epoch 200: Loss = 7.3674
Epoch 300: Loss = 7.3440
Epoch 400: Loss = 7.3301
Epoch 500: Loss = 7.3219
Epoch 600: Loss = 7.3171
Epoch 700: Loss = 7.3142
Epoch 800: Loss = 7.3126
Epoch 900: Loss = 7.3116


In [23]:
test_preds = forward(X_test, w, b)
test_mse = mse(y_test, test_preds)
test_mae = np.mean(np.abs(y_test - test_preds))

print(f"Test MSE:{test_mse:.4f}")
print(f"Test MAE:{test_mae:.4f} (Average error in years)")

Test MSE:5.3816
Test MAE:1.8009 (Average error in years)


In [24]:
#Predictions of samples
print(f"{'True Age':<10} | {'Predicted':<10} | {'Abs Error':<10}")
print("-" * 35)
for i in range(5):
    err = abs(y_test[i] - test_preds[i])
    print(f"{y_test[i]:<10.1f} | {test_preds[i]:<10.1f} | {err:<10.2f}")

True Age   | Predicted  | Abs Error 
-----------------------------------
13.5       | 11.3       | 2.25      
15.5       | 10.0       | 5.54      
14.5       | 10.8       | 3.74      
14.5       | 10.8       | 3.68      
13.5       | 11.1       | 2.43      


In [ ]:

# Systematic Errors / Bias:
# The model likely gets the very old abalones wrong (usually guessing too low).
# Why?
# 1. Our Model is a STRAIGHT LINE (y = wx + b). It assumes growth is constant forever.
# 2. Biology is a CURVE. Abalones grow fast when young, but stop growing when old.
# 3. Since our model cannot "bend" to match that curve, it fails on the older ones.